<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/Proximal_Policy_Optimization_(PPO)_%5BOn_Policy_Policy_Gradient%5D_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PPO (Proximal Policy Optimization) — Policy Gradient Method

O **PPO (Proximal Policy Optimization)** é um dos algoritmos mais utilizados atualmente em **Deep Reinforcement Learning** dentro da família dos **Policy Gradient Methods**.

Ele foi proposto por pesquisadores da OpenAI em 2017 como uma alternativa mais simples e prática ao **TRPO (Trust Region Policy Optimization)**.

A ideia principal do PPO é:

> **Atualizar a política para melhorar o comportamento do agente, mas evitando mudanças exageradamente grandes em uma única atualização.**

Em outras palavras:

**PPO tenta obter a estabilidade do TRPO com uma implementação muito mais simples.**

---

## 1. Relembrando Policy Gradient

Em Reinforcement Learning temos um agente que interage com um ambiente:

```
Estado s_t
    ↓
Política π_θ
    ↓
 Ação a_t
    ↓
 Ambiente
    ↓
Recompensa r_t
```

A política é uma rede neural parametrizada:

$$
\pi_\theta(a|s)
$$

onde:

* ($s$) = estado observado
* ($a$) = ação escolhida
* ($\theta$) = pesos da rede neural

O objetivo é maximizar a recompensa acumulada:

$$
J(\theta)=E
\left[
\sum_{t=0}^{T}
\gamma^t r_t
\right]
$$

---

## 2. O Problema dos Policy Gradients Tradicionais

No Policy Gradient clássico:

$$
\theta_{novo}=\theta_{antigo}
+
\alpha\nabla_\theta J(\theta)
$$

O agente faz uma atualização baseada no gradiente.

Porém existe um problema:

Imagine que a política atual aprendeu que:

```
Estado:
     ↓
Ação A = boa
```

Depois de uma grande atualização:

```
Estado:
     ↓
Ação B = escolhida
```

A política pode mudar tanto que ela esquece comportamentos úteis.

Esse fenômeno é chamado:

**policy collapse** ou degradação da política.

---

## 3. A Ideia que Veio do TRPO

O TRPO introduziu a ideia:

> "Não deixe a nova política ficar muito distante da antiga."

Ele usava uma restrição baseada na divergência KL:

$$
D_{KL}
(\pi_{old}||\pi_{new})
<
\delta
$$

Isso funciona bem, mas é matematicamente complexo.

O TRPO precisa de:

* Hessiana aproximada
* Conjugate Gradient
* Otimização restrita

O PPO busca uma solução mais simples.

---

## 4. Ideia Fundamental do PPO

O PPO troca a restrição rígida do TRPO por uma penalização simples.

Ele pergunta:

> "Quanto a nova política mudou em relação à antiga?"

Essa mudança é medida pelo:

### Probability Ratio

$$
r_t(\theta)=\frac{
\pi_\theta(a_t|s_t)
}
{
\pi_{old}(a_t|s_t)
}
$$

---

## 5. Interpretando o Ratio

O valor:

$$
r_t(\theta)
$$

mostra a alteração da política.

Exemplos:

### Caso 1

$$
r_t=1
$$

A política não mudou.

### Caso 2

$$
r_t=1.2
$$

A ação ficou 20% mais provável.

### Caso 3

$$
r_t=0.5
$$

A ação ficou metade da probabilidade anterior.

---

## 6. A Função Objetivo do PPO

O Policy Gradient tradicional usa:

$$
L(\theta)=r_t(\theta)A_t
$$

onde:

$$
A_t
$$

é a vantagem (*advantage*).

A vantagem responde:

> "Essa ação foi melhor ou pior que o esperado?"

---

## 7. O Grande Truque do PPO: Clipping

O PPO limita o quanto o ratio pode mudar.

A função objetivo fica:

$$
L^{CLIP}=min
(
r_t(\theta)A_t,
clip(r_t(\theta),1-\epsilon,1+\epsilon)A_t
)
$$

---

### O que significa o clipping?

O PPO define uma faixa:

$$
1-\epsilon
\leq r_t
\leq
1+\epsilon
$$

Por exemplo:

Se:

$$
\epsilon=0.2
$$

então:

$$
0.8
\leq r_t
\leq1.2
$$

A política pode mudar no máximo 20%.

---

## 8. Exemplo Intuitivo

Imagine:

A política antiga:

```
Mover para direita:
70%
Mover para esquerda:
30%
```

Depois do treinamento:

Nova política:

```
Mover direita:
99%
Mover esquerda:
1%
```

O PPO diz:

"Essa mudança é muito grande!"

Ele limita:

```
Mover direita:
85%
Mover esquerda:
15%
```

Assim evita uma atualização destrutiva.

---

## 9. Como o PPO Aprende?

O ciclo de treinamento:

```
1. Executa a política atual

        ↓

2. Coleta trajetórias

        ↓

3. Calcula recompensas

        ↓

4. Calcula Advantage

        ↓

5. Atualiza a rede usando PPO

        ↓

6. Repete
```

---

## 10. Advantage Function

O PPO geralmente utiliza:

$$
A(s,a)=
Q(s,a)-V(s)
$$

onde:

* (Q(s,a)) indica o valor da ação
* (V(s)) indica o valor médio esperado

Interpretação:

Se:

$$
A>0
$$

A ação foi melhor que o esperado.

Se:

$$
A<0
$$

A ação foi pior.

---

## 11. PPO Actor-Critic

Na prática, PPO usa arquitetura:

```
           Rede Neural

              |
       -----------------
       |               |
     Actor           Critic
       |               |
  Política πθ       Valor V(s)
```

### Actor

Aprende:

$$
\pi_\theta(a|s)
$$

### Critic

Estima:

$$
V(s)
$$

---

## 12. Generalized Advantage Estimation (GAE)

O PPO normalmente usa GAE:

$$
A_t^{GAE}=\sum
(\gamma\lambda)^k
\delta_{t+k}
$$

onde:

$$
\delta_t=r_t+\gamma V(s_{t+1})-V(s_t)
$$

GAE reduz:

* variância
* instabilidade
* ruído

---

## 13. Função Completa de Perda do PPO

O PPO normalmente combina:

### Política

$$
L_{policy}
$$

### Valor

$$
L_{value}
$$

### Entropia

$$
S[\pi]
$$

A perda final:

$$
L
=
L_{policy}
+
c_1L_{value}
-
c_2S
$$

---

## 14. Por que adicionar Entropia?

Sem entropia:

O agente pode ficar preso cedo.

Exemplo:

```
Ação esquerda: 100%
Ação direita: 0%
```

A entropia incentiva exploração:

```
Ação esquerda: 80%
Ação direita: 20%
```

Isso mantém **exploration**.

---

## 15. PPO versus TRPO

| Característica     | TRPO     | PPO           |
| ------------------ | -------- | ------------- |
| Restrição KL       | Sim      | Não explícita |
| Clipping           | Não      | Sim           |
| Hessiana           | Sim      | Não           |
| Conjugate Gradient | Sim      | Não           |
| Implementação      | Complexa | Simples       |
| Uso atual          | Menor    | Muito alto    |

---

## 16. PPO em ambientes contínuos

PPO é muito usado em:

* robótica
* controle
* veículos autônomos
* simulação física

Exemplos:

* MuJoCo
* OpenAI Gym

Tarefas:

* caminhar
* controlar braços robóticos
* equilíbrio
* navegação

---

## 17. Intuição Final

Imagine ensinar alguém a dirigir.

Policy Gradient tradicional:

> "Corrija o aluno mudando tudo de uma vez."

TRPO:

> "Só permita mudanças dentro de uma região segura."

PPO:

> "Faça mudanças pequenas e corte qualquer correção exagerada."

Por isso o PPO se tornou um dos algoritmos padrão em Deep Reinforcement Learning.

---

## Resumo em uma frase

**PPO é um método de Policy Gradient que melhora a política usando uma função objetivo com clipping, evitando atualizações grandes demais e proporcionando treinamento estável e eficiente.**


## Estrutura do código exemplo

### 1. `ActorCritic` — A rede neural
Tronco compartilhado com duas cabeças:
- **Actor** → distribui probabilidade sobre ações (política π)
- **Critic** → estima o valor do estado V(s) (baseline)

Usa inicialização ortogonal, que é a prática padrão para PPO.

### 2. `RolloutBuffer` — Coleta de experiência
Armazena as trajetórias coletadas e calcula o **GAE** (Generalized Advantage Estimation) de trás pra frente:

$$A_t = \delta_t + \gamma\lambda(1-d_t)\cdot A_{t+1}$$

O GAE reduz a variância do gradiente com λ controlando o tradeoff viés/variância.

### 3. `PPOAgent.update()` — O coração do PPO
Roda **N_EPOCHS** vezes sobre o buffer com mini-batches aleatórios, calculando a perda clipping:

$$L = -\underbrace{\min(r\cdot A,\ \text{clip}(r,1\pm\varepsilon)\cdot A)}_{\text{PPO-Clip}} + \underbrace{0.5\cdot\text{MSE}(V, R)}_{\text{Critic}} - \underbrace{0.01\cdot H[\pi]}_{\text{Entropia}}$$

### 4. Loop de treino
Usa `gym.vector.SyncVectorEnv` com **4 ambientes paralelos** para coletar experiências mais diversas por iteração.

---

## Como rodar

```bash
pip install torch gymnasium pygame

# Treino (CUDA automático se disponível)
python ppo_pytorch.py
```

Para usar em outro ambiente, basta trocar `ENV_NAME` e, se for contínuo, substituir `Categorical` por `Normal` na rede.

### Novos ambientes — `--env`
| Ambiente | Tipo de ação | Dificuldade |
|---|---|---|
| `CartPole-v1` | Discreto | Fácil |
| `Acrobot-v1` | Discreto | Médio |
| `Pendulum-v1` | **Contínuo** | Médio |
| `Hopper-v4` | **Contínuo** | Difícil (MuJoCo) |

Cada ambiente tem seus próprios hiperparâmetros configurados em `ENV_CONFIGS`.

### Suporte a ações contínuas
A `ActorCritic` agora escolhe automaticamente entre `Categorical` (discreto) e `Normal` (contínuo). Para contínuo:
- Actor produz **μ(s)** e um `log_std` treinável global
- `log_prob` é somado sobre as dimensões da ação

### Gráficos — `MetricsLogger.plot()`
Painel 3×2 salvo automaticamente como PNG:

| Posição | Métrica | O que diagnóstica |
|---|---|---|
| [0,0] | **Retorno** (+ banda min/max) | Aprendizado geral |
| [0,1] | **Clip Fraction** | Saúde do clipping PPO |
| [1,0] | **Policy Loss** | Estabilidade do actor |
| [1,1] | **Value Loss** | Precisão do critic |
| [2,0] | **Entropia H[π]** | Nível de exploração |
| [2,1] | **Steps/segundo** | Throughput de coleta |

Todas as curvas têm suavização por média móvel.

---

## Como rodar

```bash
# Instalar dependências
pip install torch gymnasium matplotlib

# Treinar (CartPole padrão)
python ppo_pytorch.py

# Outros ambientes
python ppo_pytorch.py --env Acrobot-v1
python ppo_pytorch.py --env Pendulum-v1
python ppo_pytorch.py --env Hopper-v4   # requer: pip install gymnasium[mujoco]

# Com avaliação visual ao final
python ppo_pytorch.py --env Pendulum-v1 --eval

# Reproduzibilidade
python ppo_pytorch.py --env CartPole-v1 --seed 0
```

O gráfico é salvo automaticamente como `ppo_<env>_metrics.png` na pasta atual.